In [1]:
import pysacsnow as pss
import pandas as pd
import time
from pathlib import Path
import numpy as np

## Batch Model Runs with run() — Performance & Correctness Demo

This notebook demonstrates the new run() batch method available on every process class (SNOW17, SACSMA, UHG, CONSUSE).

### What this notebook covers  
1. Baseline (numpy, step-by-step loop) — The classic advance() / calculate() / output() loop using pure-numpy kernels. Used as the correctness reference.
2. Batch run with numba — The same model chain driven by .run(), which internally dispatches each model's entire time loop to a single Numba-compiled @njit function. Significantly faster, especially on second and subsequent calls when Numba's disk cache is warm.
3. Output comparison — Verifies that both approaches produce numerically identical results within tight tolerances.
4. Reset + re-run — Shows the .reset() method, which rewinds model state to time-zero without re-constructing or re-compiling anything, then demonstrates a second run with a modified parameter — the intended workflow for calibration loops.

### Key concepts
| Method                         | When to use                                                                 |
|--------------------------------|------------------------------------------------------------------------------|
| `model.run(finalize=False)`    | Full timeseries batch run; returns buffer_states dict of (n_times, nhru) arrays |
| `model.reset()`                | Rewind state to t=0 so the same instance can be re-run with new parameters   |
| `input_overrides={"var": array}` | Pass (n_times, nhru) arrays to replace adapter-read forcing (e.g. routing SNOW17 output into SACSMA) |


In [2]:
# Write states
basin_name = "16103000"

yaml_file = Path(f"../configurations/sac/sac.control.{basin_name}.calb.yaml")
sac_control = pss.Controller.load_from_yaml(yaml_file)
yaml_file = Path(f"../configurations/unithg/uhg.control.{basin_name}.calb.yaml")
uhg_control = pss.Controller.load_from_yaml(yaml_file)

## Setup — Load controllers and instantiate models
Each model has its own Controller (which holds the simulation period, timestep, and paths to forcing/parameter files). Loading all four controllers here so they can be reused across both the baseline and batch-run sections below.

The same controllers are shared between both runs — the controller's time counters are reset automatically when .reset() is called on a model.

In [3]:
sacsma = pss.SACSMA(control=sac_control, calc_method="numpy")
uhg = pss.UHG(control=uhg_control, calc_method="numpy")

## Section 1 — Baseline: step-by-step loop (numpy)  
The traditional way to run the model chain: loop over every timestep, manually calling advance() → calculate() → output() on each model. Outputs from one model are fed directly into the next at each step (SNOW17 → SACSMA → UHG). CONSUSE runs separately at daily resolution after the hourly chain completes.

This is the correctness reference — the numba batch results below are compared against these outputs.

In [ ]:
sacsma.reset()
uhg.reset()

%%time
# Run models
for tt in range(sac_control.n_times):

    sac_control.advance()
    sacsma.advance()
    sacsma.calculate()
    sacsma.output()

    uhg_control.advance()
    uhg.advance()
    uhg.hru_tci[:] = sacsma.tci[:]
    uhg.calculate()
    uhg.output()

basin_tci = sacsma.buffer_states["tci"]
total_flow = np.sum(uhg.buffer_states["q"], axis=1)
flow_df = pd.DataFrame({"flow_sim": total_flow}, index=uhg.value_time_state)

In [ ]:
flow_df_fin = flow_df.copy()

## Section 2 — Batch run with run() and Numba
Instantiate the same models with calc_method="numba". On the first call, Numba JIT-compiles the inner calculation kernel and caches it to disk. Subsequent calls (in this session or future sessions) load directly from the cache and are significantly faster.

Rather than an explicit timestep loop, each model's entire timeseries is executed in a single model.run(finalize=False) call. Outputs from one model are passed to the next via input_overrides, which accepts a (n_times, nhru) numpy array for any input variable.

In [ ]:
# Load with numba
sacsma = pss.SACSMA(control=sac_control, calc_method="numba")
uhg = pss.UHG(control=uhg_control, calc_method="numba")

%%time
# Execute the same sequence of model runs, but with numba and run() methods.

# Sac
sacsma.run(finalize=False)
keys = ["tci"]
uhg_input = {k: sacsma.buffer_states[k] for k in keys}

# UHG
uhg.run(finalize=False, input_overrides={"hru_tci": uhg_input["tci"]})
total_flow = np.sum(uhg.buffer_states["q"], axis=1)
flow_df_run = pd.DataFrame({"flow_sim": total_flow}, index=uhg.value_time_state)
     

In [ ]:
flow_df_run

In [ ]:
flow_df_fin